In [4]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import os
import nibabel as nib
import pandas as pd

import warnings


In [5]:
input_mm = 0.2   # only used to locate the input folder; real voxel size is read from each header
output_mm = 0.6  # target voxel size: scalar (isotropic) or (x, y, z) tuple

HERE = os.path.abspath("")  # notebook directory (__data_test__/inputs)
INPUT_RECON_DIR = os.path.join(HERE, f"complete/changed_recon_{input_mm}mm")
CHANGED_INPUT_RECON_DIR = os.path.join(HERE, f"complete/changed_recon_{output_mm}mm")
case_list = sorted(os.listdir(INPUT_RECON_DIR))

os.makedirs(CHANGED_INPUT_RECON_DIR, exist_ok=True)


def mode_downsample(labels, f):
    """Downsample a label map by integer factors f=(fx, fy, fz) using majority vote over each block."""
    fx, fy, fz = f
    # pad with background so every dimension is a multiple of its factor
    pad = [(0, (-s) % k) for s, k in zip(labels.shape, f)]
    labels = np.pad(labels, pad, mode="constant", constant_values=0)
    X, Y, Z = (s // k for s, k in zip(labels.shape, f))
    out = np.zeros((X, Y, Z), dtype=labels.dtype)
    for x in range(X):  # slab by slab to keep memory bounded
        slab = labels[x * fx:(x + 1) * fx]
        blocks = slab.reshape(fx, Y, fy, Z, fz).transpose(1, 3, 0, 2, 4).reshape(Y, Z, fx * fy * fz)
        # count, for each voxel in the block, how many voxels share its label; pick the most frequent
        counts = (blocks[..., :, None] == blocks[..., None, :]).sum(-1)
        out[x] = np.take_along_axis(blocks, counts.argmax(-1)[..., None], -1)[..., 0]
    return out


def nearest_resample(labels, scale, new_shape):
    """Nearest-neighbour resampling of a label map onto a grid with voxel size scale * input voxel size."""
    out = np.zeros(new_shape, dtype=labels.dtype)
    # output voxel i has its centre at input voxel coordinate (i + 0.5) * scale - 0.5
    idx = [np.clip(np.round((np.arange(n) + 0.5) * s - 0.5).astype(int), 0, S - 1)
           for n, s, S in zip(new_shape, scale, labels.shape)]
    for x, ix in enumerate(idx[0]):
        out[x] = labels[ix][np.ix_(idx[1], idx[2])]
    return out


def resample_labels(img, out_mm):
    """Resample a NIfTI label map to voxel size out_mm, keeping the same field of view."""
    labels = np.asarray(img.dataobj)  # keep integer labels (no float conversion)
    in_mm = np.round(np.array(img.header.get_zooms()[:3], dtype=float), 6)  # drop float32 noise (0.2 -> 0.20000000298)
    out_mm = np.broadcast_to(np.asarray(out_mm, dtype=float), (3,))
    scale = out_mm / in_mm  # >1 downsample, <1 upsample
    new_shape = tuple(int(np.ceil(s / k - 1e-6)) for s, k in zip(labels.shape, scale))

    f = np.round(scale).astype(int)
    if np.allclose(scale, f) and (f >= 1).all():
        new_data = mode_downsample(labels, tuple(f))  # integer factor: majority vote
    else:
        new_data = nearest_resample(labels, scale, new_shape)  # arbitrary factor: nearest neighbour

    # scale voxel axes and shift the origin to the centre of the first output voxel
    new_affine = img.affine.copy()
    new_affine[:3, :3] = img.affine[:3, :3] * scale
    new_affine[:3, 3] = img.affine[:3, :3] @ (0.5 * scale - 0.5) + img.affine[:3, 3]

    new_img = nib.Nifti1Image(new_data, new_affine, img.header)
    new_img.set_data_dtype(img.get_data_dtype())
    new_img.header.set_zooms(tuple(out_mm))
    return new_img


for case in case_list:
    if case.endswith(".nii.gz"):
        print(case)
        img = nib.load(os.path.join(INPUT_RECON_DIR, case))
        new_img = resample_labels(img, output_mm)
        nib.save(new_img, os.path.join(CHANGED_INPUT_RECON_DIR, case))
        print(f"  {img.shape} @ {img.header.get_zooms()} -> {new_img.shape} @ {new_img.header.get_zooms()}")
        print(f"  unique labels: {len(np.unique(new_img.get_fdata()))}")


case-1.nii.gz
  (467, 923, 589) @ (np.float32(0.2), np.float32(0.2), np.float32(0.2)) -> (156, 308, 197) @ (np.float32(0.6), np.float32(0.6), np.float32(0.6))
case-2.nii.gz
  (489, 928, 624) @ (np.float32(0.2), np.float32(0.2), np.float32(0.2)) -> (163, 310, 208) @ (np.float32(0.6), np.float32(0.6), np.float32(0.6))
case-3.nii.gz
  (667, 1024, 869) @ (np.float32(0.2), np.float32(0.2), np.float32(0.2)) -> (223, 342, 290) @ (np.float32(0.6), np.float32(0.6), np.float32(0.6))
case-4.nii.gz
  (468, 977, 730) @ (np.float32(0.2), np.float32(0.2), np.float32(0.2)) -> (156, 326, 244) @ (np.float32(0.6), np.float32(0.6), np.float32(0.6))
case-5.nii.gz
  (446, 898, 649) @ (np.float32(0.2), np.float32(0.2), np.float32(0.2)) -> (149, 300, 217) @ (np.float32(0.6), np.float32(0.6), np.float32(0.6))


In [6]:
for case in case_list:
    if case.endswith(".nii.gz"):
        print(case)
        img = nib.load(os.path.join(CHANGED_INPUT_RECON_DIR, case))
        new_img = resample_labels(img, output_mm)

        unique_labels = np.unique(new_img.get_fdata())
        print(f"  {img.shape} @ {img.header.get_zooms()} -> {new_img.shape} @ {new_img.header.get_zooms()} (unique labels: {len(unique_labels)})")

case-1.nii.gz
  (156, 308, 197) @ (np.float32(0.6), np.float32(0.6), np.float32(0.6)) -> (156, 308, 197) @ (np.float32(0.6), np.float32(0.6), np.float32(0.6)) (unique labels: 299)
case-2.nii.gz
  (163, 310, 208) @ (np.float32(0.6), np.float32(0.6), np.float32(0.6)) -> (163, 310, 208) @ (np.float32(0.6), np.float32(0.6), np.float32(0.6)) (unique labels: 297)
case-3.nii.gz
  (223, 342, 290) @ (np.float32(0.6), np.float32(0.6), np.float32(0.6)) -> (223, 342, 290) @ (np.float32(0.6), np.float32(0.6), np.float32(0.6)) (unique labels: 297)
case-4.nii.gz
  (156, 326, 244) @ (np.float32(0.6), np.float32(0.6), np.float32(0.6)) -> (156, 326, 244) @ (np.float32(0.6), np.float32(0.6), np.float32(0.6)) (unique labels: 295)
case-5.nii.gz
  (149, 300, 217) @ (np.float32(0.6), np.float32(0.6), np.float32(0.6)) -> (149, 300, 217) @ (np.float32(0.6), np.float32(0.6), np.float32(0.6)) (unique labels: 297)
